In [0]:
import requests
import json
import os
import csv
import math

In [0]:
season = "2025"
round_no = "2"

batch_id = f"{season}-{round_no.zfill(2)}"

### Results

In [0]:
results_response = requests.get(f'https://api.jolpi.ca/ergast/f1/{season}/{round_no}/results/')
results_data = results_response.json()

race = results_data['MRData']['RaceTable']['Races'][0]

records = []
for result in race['Results']:
    records.append({
        "date": race['date'],
        "raceName": race['raceName'].lower(),
        "round": race['round'],
        "season": race['season'],
        "url": race['url'],
        "constructorId": result['Constructor']['constructorId'],
        "driverId": result['Driver']['driverId'],
        "grid": result['grid'],
        "laps": result['laps'],
        "number": result['number'],
        "points": result['points'],
        "position": result['position'],
        "positionText": result['positionText'],
        "status": result['status']
    })

file_path = '/Volumes/formula1_incr/landing/files/demo'

output_dir = os.path.join(file_path, batch_id, 'results')
os.makedirs(output_dir, exist_ok=True)

file_name = f"results_{race['season']}.json"

with open(os.path.join(output_dir, file_name), 'w') as f:
    for record in records:
        f.write(json.dumps(record) + '\n')

print(f"Saved {len(records)} records to {output_dir}/{file_name}")

### Sprints

In [0]:
sprint_response = requests.get(f'https://api.jolpi.ca/ergast/f1/{season}/{round_no}/sprint/')
sprint_data = sprint_response.json()

races = sprint_data['MRData']['RaceTable']['Races']

records = []

if races:
    sprint_race = races[0]
    for result in sprint_race['SprintResults']:
        records.append({
            "date": sprint_race['date'],
            "raceName": sprint_race['raceName'].lower(),
            "round": int(sprint_race['round']),
            "season": int(sprint_race['season']),
            "url": sprint_race['url'],
            "constructorId": result['Constructor']['constructorId'],
            "driverId": result['Driver']['driverId'],
            "grid": int(result['grid']),
            "laps": int(result['laps']),
            "number": int(result['number']),
            "points": float(result['points']),
            "position": int(result['position']),
            "positionText": result['positionText'],
            "status": result['status']
        })
else:
    print("No sprint data for this round, writing empty file")

file_path = '/Volumes/formula1_incr/landing/files/demo'

output_dir = os.path.join(file_path, batch_id, 'sprints')
os.makedirs(output_dir, exist_ok=True)

file_name = f"sprints_{season}.json"

with open(os.path.join(output_dir, file_name), 'w') as f:
    if records:
        json.dump(records, f, indent=2)
    else:
        f.write('[]')

print(f"Saved {len(records)} records to {output_dir}/{file_name}")

### Circuits

In [0]:
circuits_response = requests.get('https://api.jolpi.ca/ergast/f1/circuits/?limit=100')
circuits_data = circuits_response.json()

circuits = circuits_data['MRData']['CircuitTable']['Circuits']

records = []
for circuit in circuits:
    records.append({
        "circuitId": circuit['circuitId'],
        "url": circuit['url'],
        "circuitName": circuit['circuitName'].lower(),
        "lat": circuit['Location']['lat'],
        "long": circuit['Location']['long'],
        "locality": circuit['Location']['locality'].lower(),
        "country": circuit['Location']['country']
    })

file_path = '/Volumes/formula1_incr/landing/files/demo'

output_dir = os.path.join(file_path, batch_id)
os.makedirs(output_dir, exist_ok=True)

file_name = 'circuits.csv'

with open(os.path.join(output_dir, file_name), 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=["circuitId", "url", "circuitName", "lat", "long", "locality", "country"])
    writer.writeheader()
    writer.writerows(records)

print(f"Saved {len(records)} records to {output_dir}/{file_name}")

### Races

In [0]:
def race_requests(limit, offset):
    races_response = requests.get(f'https://api.jolpi.ca/ergast/f1/races/?limit={limit}&offset={offset}')
    return races_response.json()

limit = 100

first_response = race_requests(limit, 0)
total = int(first_response['MRData']['total'])
total_pages = math.ceil(total / limit)

print(f"Total races: {total}, Pages: {total_pages}")

records = []

for page in range(0, total_pages):
    races_data = race_requests(limit, page * limit) if page > 0 else first_response
    for race in races_data['MRData']['RaceTable']['Races']:
        records.append({
            "season": race['season'],
            "round": race['round'],
            "url": race['url'],
            "raceName": race['raceName'].lower(),
            "date": race['date'],
            "circuitId": race['Circuit']['circuitId']
        })

file_path = '/Volumes/formula1_incr/landing/files/demo'

output_dir = os.path.join(file_path, batch_id)
os.makedirs(output_dir, exist_ok=True)

file_name = 'races.csv'

with open(os.path.join(output_dir, file_name), 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=["season", "round", "url", "raceName", "date", "circuitId"])
    writer.writeheader()
    writer.writerows(records)

print(f"Saved {len(records)} records to {output_dir}/{file_name}")

### Constructors

In [0]:
def constructors_requests(limit, offset):
    constructors_response = requests.get(f'https://api.jolpi.ca/ergast/f1/constructors/?limit={limit}&offset={offset}')
    return constructors_response.json()

limit = 100

first_response = constructors_requests(limit, 0)
total = int(first_response['MRData']['total'])
total_pages = math.ceil(total / limit)

print(f"Total constructors: {total}, Pages: {total_pages}")

records = []

for page in range(0, total_pages):
    constructors_data = constructors_requests(limit, page * limit) if page > 0 else first_response
    for constructor in constructors_data['MRData']['ConstructorTable']['Constructors']:
        records.append({
            "constructorId": constructor['constructorId'],
            "name": constructor['name'],
            "nationality": constructor['nationality'].lower(),
            "url": constructor.get('url', '')
        })

file_path = '/Volumes/formula1_incr/landing/files/demo'

output_dir = os.path.join(file_path, batch_id)
os.makedirs(output_dir, exist_ok=True)

file_name = 'constructors.json'

with open(os.path.join(output_dir, file_name), 'w') as f:
    for record in records:
        f.write(json.dumps(record) + '\n')

print(f"Saved {len(records)} records to {output_dir}/{file_name}")

### Drivers

In [0]:
def drivers_requests(limit, offset):
    drivers_response = requests.get(f'https://api.jolpi.ca/ergast/f1/drivers/?limit={limit}&offset={offset}')
    return drivers_response.json()

limit = 100

first_response = drivers_requests(limit, 0)
total = int(first_response['MRData']['total'])
total_pages = math.ceil(total / limit)

print(f"Total drivers: {total}, Pages: {total_pages}")

records = []

for page in range(0, total_pages):
    drivers_data = drivers_requests(limit, page * limit) if page > 0 else first_response
    for driver in drivers_data['MRData']['DriverTable']['Drivers']:
        records.append({
            "driverId": driver['driverId'],
            "name": {
                "givenName":driver['givenName'].lower(),
                "familyName":driver['familyName'].lower()
                },
            "dateOfBirth":driver.get('dateOfBirth', ''),
            "nationality": driver.get('nationality', '').lower(),
            "url": driver.get('url', '')
        })

file_path = '/Volumes/formula1_incr/landing/files/demo'

output_dir = os.path.join(file_path, batch_id)
os.makedirs(output_dir, exist_ok=True)

file_name = 'drivers.json'

with open(os.path.join(output_dir, file_name), 'w') as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"Saved {len(records)} records to {output_dir}/{file_name}")